# Milestone 2: Understanding & Reasoning Engine

## Task B2: Understanding & Reasoning Engine [20 Marks]

This notebook implements the core cognitive capabilities for understanding and reasoning with business intelligence data.

### Objectives:
- Implement sentiment analysis (Understanding)
- Build knowledge graphs for reasoning
- Train ML models for classification/prediction
- Integrate multiple cognitive capabilities


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Import custom modules
import sys
sys.path.append('../../')
from src.models.sentiment_analyzer import SentimentAnalyzer
from src.utils.text_preprocessor import TextPreprocessor

print("Libraries imported successfully!")


## Part 1: Understanding - Sentiment Analysis

Implement Natural Language Processing to understand customer sentiment from unstructured text.


In [ ]:
# Load processed data
try:
    df_reviews = pd.read_csv('../../data/processed/cleaned_reviews.csv')
    print(f"✅ Loaded {len(df_reviews)} processed reviews")
except FileNotFoundError:
    print("⚠️ Processed data not found. Please run Milestone 1 notebook first.")
    # Create sample data for demonstration
    df_reviews = pd.DataFrame({
        'text': ["Great service, very fast delivery", "Product was damaged", 
                 "Amazing quality, will buy again", "Late delivery but good product"] * 25,
        'cleaned_text': ["great service fast delivery", "product damaged",
                        "amazing quality buy again", "late delivery good product"] * 25
    })

df_reviews.head()


In [ ]:
# Initialize sentiment analyzer
analyzer = SentimentAnalyzer(method='vader')

# Analyze sentiment for all reviews
print("Analyzing sentiment for all reviews...")
sentiment_results = []

for text in df_reviews['cleaned_text']:
    if text and len(text) > 0:
        result = analyzer.analyze(text)
        sentiment_results.append(result)
    else:
        sentiment_results.append({'sentiment': 'neutral', 'compound': 0.0})

# Create sentiment dataframe
df_sentiment = pd.DataFrame(sentiment_results)
df_reviews_with_sentiment = pd.concat([df_reviews, df_sentiment], axis=1)

print("✅ Sentiment analysis completed!")
print(f"\nSentiment distribution:")
print(df_reviews_with_sentiment['sentiment'].value_counts())

df_reviews_with_sentiment.head()


In [ ]:
# Visualize sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sentiment counts
sentiment_counts = df_reviews_with_sentiment['sentiment'].value_counts()
axes[0].bar(sentiment_counts.index, sentiment_counts.values, color=['green', 'red', 'gray'])
axes[0].set_title('Sentiment Distribution')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(sentiment_counts.index, rotation=45)

# Compound score distribution
if 'compound' in df_reviews_with_sentiment.columns:
    axes[1].hist(df_reviews_with_sentiment['compound'], bins=30, edgecolor='black', color='skyblue')
    axes[1].set_title('Compound Sentiment Score Distribution')
    axes[1].set_xlabel('Compound Score')
    axes[1].set_ylabel('Frequency')
    axes[1].axvline(x=0, color='red', linestyle='--', label='Neutral')
    axes[1].legend()

plt.tight_layout()
plt.show()


## Part 2: Reasoning - Topic Modeling

Extract key topics from customer feedback to understand main themes and concerns.


In [ ]:
# Topic Modeling using LDA
from gensim import corpora, models
from gensim.models import LdaModel
from gensim.utils import simple_preprocess
from collections import Counter

# Prepare documents for topic modeling
documents = [text.split() for text in df_reviews_with_sentiment['cleaned_text'] if text and len(text) > 0]

# Create dictionary and corpus
dictionary = corpora.Dictionary(documents)
dictionary.filter_extremes(no_below=2, no_above=0.5)  # Filter rare and common words
corpus = [dictionary.doc2bow(doc) for doc in documents]

print(f"✅ Prepared {len(documents)} documents for topic modeling")
print(f"Dictionary size: {len(dictionary)} unique words")
